In [4]:
import pandas as pd
from google.colab import files

print("Upload all 4 files: AAPL_predictions_single.csv, JPM_predictions.csv, XOM_predictions.csv, JNJ_predictions.csv")
uploaded = files.upload()

dfs = []
for fname in uploaded.keys():
    df = pd.read_csv(fname)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    dfs.append(df.set_index('Timestamp'))

combined = pd.concat(dfs, axis=1)
combined = combined.reset_index()

print(f"Combined shape: {combined.shape}")
print(combined.columns.tolist())
print(combined.head())

combined.to_csv('combined_predictions.csv', index=False)

Upload all 4 files: AAPL_predictions_single.csv, JPM_predictions.csv, XOM_predictions.csv, JNJ_predictions.csv


Saving AAPL_predictions_single.csv to AAPL_predictions_single (1).csv
Saving JNJ_predictions.csv to JNJ_predictions (1).csv
Saving XOM_predictions.csv to XOM_predictions (1).csv
Saving JPM_predictions.csv to JPM_predictions (1).csv
Combined shape: (150, 5)
['Timestamp', 'AAPL', 'JNJ', 'XOM', 'JPM']
                  Timestamp        AAPL         JNJ         XOM         JPM
0 2026-09-18 20:00:00+00:00  335.732839  270.145599  162.836328  349.171541
1 2026-09-18 20:01:00+00:00  335.653156  270.322455  162.718356  349.529197
2 2026-09-18 20:02:00+00:00  335.604745  270.488315  162.622189  349.753873
3 2026-09-18 20:03:00+00:00  335.633870  270.487543  162.556894  349.873701
4 2026-09-18 20:04:00+00:00  335.657565  270.505491  162.511831  349.843837


In [5]:
files.download('combined_predictions.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
import pandas as pd
import numpy as np
from google.colab import files

# Step 1: Upload and load dataset
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

# Step 2: Validate shape
num_rows, num_cols = df.shape
if num_rows < 2 or num_cols < 2:
    raise ValueError(f"Dataset must have at least 2 rows and 2 columns, got {df.shape}")

# FIX: original required the first column to be sequential integers
# (1, 2, 3...) — our data uses real timestamps instead, so we validate
# that it's a parseable, sorted datetime column instead.
df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0], errors='coerce')
if df.iloc[:, 0].isna().any():
    raise ValueError("Timestamp column contains values that could not be parsed as dates.")
if not df.iloc[:, 0].is_monotonic_increasing:
    raise ValueError("Timestamp column must be sorted in increasing order.")

# Rename first column to 'timestamp' for consistency
df.columns = ['timestamp'] + list(df.columns[1:])

print(f"Loaded {filename}: {df.shape}")
print(df.head())

Saving combined_predictions.csv to combined_predictions (1).csv
Loaded combined_predictions (1).csv: (150, 5)
                   timestamp        AAPL         JNJ         XOM         JPM
0  2026-09-18 20:00:00+00:00  335.732839  270.145599  162.836328  349.171541
1  2026-09-18 20:01:00+00:00  335.653156  270.322455  162.718356  349.529197
2  2026-09-18 20:02:00+00:00  335.604745  270.488315  162.622189  349.753873
3  2026-09-18 20:03:00+00:00  335.633870  270.487543  162.556894  349.873701
4  2026-09-18 20:04:00+00:00  335.657565  270.505491  162.511831  349.843837


/usr/local/lib/python3.13/dist-packages/pandas/core/base.py:1110: FutureWarning: Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Index constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.
  return Index(self).is_monotonic_increasing


In [9]:
# Step 2: Analyze each price column (all columns except timestamp)
price_columns = df.columns[1:]  # All columns except timestamp
results = []

for col in price_columns:
    prices = df[col]
    max_price = prices.max()
    min_price = prices.min()
    max_idx = prices.idxmax()
    min_idx = prices.idxmin()
    max_time = df['timestamp'].iloc[max_idx]
    min_time = df['timestamp'].iloc[min_idx]
    std_dev = prices.std()

    results.append({
        'asset': col,
        'max_price': max_price,
        'min_price': min_price,
        'max_time': max_time,
        'min_time': min_time,
        'std_dev': std_dev
    })

results_df = pd.DataFrame(results)

sell_portfolio = results_df[results_df['max_time'] < results_df['min_time']]['asset'].tolist()
buy_portfolio = results_df[results_df['max_time'] > results_df['min_time']]['asset'].tolist()

In [10]:
print("df shape:", df.shape)
print(results_df)
print(f"\nBuy portfolio (price rises: min before max): {buy_portfolio}")
print(f"Sell portfolio (price falls: max before min): {sell_portfolio}")

df shape: (150, 5)
  asset   max_price   min_price                  max_time  \
0  AAPL  335.732839  335.467923 2026-09-18 20:00:00+00:00   
1   JNJ  270.682978  270.145599 2026-09-18 20:13:00+00:00   
2   XOM  162.836328  161.983495 2026-09-18 20:00:00+00:00   
3   JPM  351.077522  349.171541 2026-09-18 20:11:00+00:00   

                   min_time   std_dev  
0 2026-09-18 22:13:00+00:00  0.047657  
1 2026-09-18 20:00:00+00:00  0.052234  
2 2026-09-18 20:15:00+00:00  0.117448  
3 2026-09-18 20:00:00+00:00  0.279903  

Buy portfolio (price rises: min before max): ['JNJ', 'JPM']
Sell portfolio (price falls: max before min): ['AAPL', 'XOM']


In [13]:
# Step 4: Calculate weights using inverse of standard deviations
results_df['inv_std'] = 1 / results_df['std_dev'].replace(0, np.inf)
total_inv_std = results_df['inv_std'].sum()
results_df['weight'] = results_df['inv_std'] / total_inv_std

# Step 5: Allocate total investment ($10,000 — increased from $1,000 so
# every asset's allocation can afford at least one share; JPM's ~$349/share
# price meant it couldn't buy in at the $1,000 budget).
total_investment = 10000

buy_weights_sum = results_df[results_df['asset'].isin(buy_portfolio)]['weight'].sum()
sell_weights_sum = results_df[results_df['asset'].isin(sell_portfolio)]['weight'].sum()

total_weight = buy_weights_sum + sell_weights_sum
if total_weight > 0:
    buy_allocation = total_investment * (buy_weights_sum / total_weight)
    sell_allocation = total_investment * (sell_weights_sum / total_weight)
else:
    buy_allocation = 0
    sell_allocation = 0

buy_weights = results_df[results_df['asset'].isin(buy_portfolio)][['asset', 'weight']].copy()
sell_weights = results_df[results_df['asset'].isin(sell_portfolio)][['asset', 'weight']].copy()

buy_weights['portfolio_weight'] = buy_weights['weight'] / buy_weights_sum if buy_weights_sum > 0 else 0
sell_weights['portfolio_weight'] = sell_weights['weight'] / sell_weights_sum if sell_weights_sum > 0 else 0

print(f"Buy allocation: ${buy_allocation:.2f}")
print(f"Sell allocation: ${sell_allocation:.2f}")

# Step 7: Calculate integer shares and balance
portfolio = []
balance = total_investment

for _, row in buy_weights.iterrows():
    if row['portfolio_weight'] > 0:
        asset = row['asset']
        min_price = results_df[results_df['asset'] == asset]['min_price'].iloc[0]
        max_price = results_df[results_df['asset'] == asset]['max_price'].iloc[0]
        allocation = buy_allocation * row['portfolio_weight']
        shares = int(allocation / min_price)
        if shares > 0:
            cost = shares * min_price
            balance -= cost
            portfolio.append({
                'asset': asset, 'type': 'buy', 'shares': shares,
                'buy_price': min_price, 'sell_price': max_price,
                'buy_time': results_df[results_df['asset'] == asset]['min_time'].iloc[0],
                'sell_time': results_df[results_df['asset'] == asset]['max_time'].iloc[0],
                'cost': cost, 'revenue': shares * (max_price - min_price)
            })

for _, row in sell_weights.iterrows():
    if row['portfolio_weight'] > 0:
        asset = row['asset']
        max_price = results_df[results_df['asset'] == asset]['max_price'].iloc[0]
        min_price = results_df[results_df['asset'] == asset]['min_price'].iloc[0]
        allocation = sell_allocation * row['portfolio_weight']
        shares = int(allocation / max_price)
        if shares > 0:
            cost = shares * max_price
            balance -= cost
            portfolio.append({
                'asset': asset, 'type': 'sell', 'shares': shares,
                'buy_price': min_price, 'sell_price': max_price,
                'buy_time': results_df[results_df['asset'] == asset]['min_time'].iloc[0],
                'sell_time': results_df[results_df['asset'] == asset]['max_time'].iloc[0],
                'cost': cost, 'revenue': shares * (max_price - min_price)
            })

portfolio_df = pd.DataFrame(portfolio)
print(portfolio_df)
print(f"\nRemaining balance: ${balance:.2f}")

Buy allocation: $4350.70
Sell allocation: $5649.30
  asset  type  shares   buy_price  sell_price                  buy_time  \
0   JNJ   buy      13  270.145599  270.682978 2026-09-18 20:00:00+00:00   
1   JPM   buy       1  349.171541  351.077522 2026-09-18 20:00:00+00:00   
2  AAPL  sell      11  335.467923  335.732839 2026-09-18 22:13:00+00:00   
3   XOM  sell      10  161.983495  162.836328 2026-09-18 20:15:00+00:00   

                  sell_time         cost   revenue  
0 2026-09-18 20:13:00+00:00  3511.892792  6.985922  
1 2026-09-18 20:11:00+00:00   349.171541  1.905981  
2 2026-09-18 20:00:00+00:00  3693.061226  2.914072  
3 2026-09-18 20:00:00+00:00  1628.363280  8.528331  

Remaining balance: $817.51


In [15]:
# Step 8: Calculate portfolio metrics (period-level, not annualized)
# NOTE: annualizing a single 150-minute forecast window would mean
# assuming this exact return repeats identically ~655 times across a
# year, which isn't a realistic claim from one simulated window. We
# report the actual period-level return/risk instead.

def calculate_portfolio_metrics(portfolio_df, portfolio_type, df_prices):
    if portfolio_type == 'buy':
        df_p = portfolio_df[portfolio_df['type'] == 'buy']
    else:
        df_p = portfolio_df[portfolio_df['type'] == 'sell']

    if df_p.empty:
        return {'return': 0, 'volatility': 0, 'sharpe_ratio': 0}

    total_cost = df_p['cost'].sum()
    total_revenue = df_p['revenue'].sum()
    period_return = total_revenue / total_cost if total_cost > 0 else 0

    period_returns = []
    for _, row in df_p.iterrows():
        if row['type'] == 'buy':
            ret = (row['sell_price'] - row['buy_price']) / row['buy_price']
        else:
            ret = (row['sell_price'] - row['buy_price']) / row['sell_price']
        weight = row['cost'] / total_cost if total_cost > 0 else 0
        period_returns.append(ret * weight)

    volatility = np.std(period_returns) if period_returns else 0
    # risk-free rate scaled down to the same 150-minute window for a fair comparison
    period_risk_free = risk_free_rate * (FORECAST_MINUTES / (MINUTES_PER_TRADING_DAY * TRADING_DAYS_PER_YEAR))
    sharpe_ratio = (period_return - period_risk_free) / volatility if volatility > 0 else 0

    return {'return': period_return, 'volatility': volatility, 'sharpe_ratio': sharpe_ratio}

risk_free_rate = 0.038
MINUTES_PER_TRADING_DAY = 390
FORECAST_MINUTES = 150
TRADING_DAYS_PER_YEAR = 252

buy_metrics = calculate_portfolio_metrics(portfolio_df, 'buy', df)
sell_metrics = calculate_portfolio_metrics(portfolio_df, 'sell', df)

total_revenue = portfolio_df['revenue'].sum()
investment_made = total_investment - balance
portfolio_final_balance = investment_made + total_revenue
overall_return = total_revenue / investment_made if investment_made > 0 else 0
overall_volatility = np.sqrt(buy_metrics['volatility']**2 + sell_metrics['volatility']**2)
period_risk_free = risk_free_rate * (FORECAST_MINUTES / (MINUTES_PER_TRADING_DAY * TRADING_DAYS_PER_YEAR))
overall_sharpe = (overall_return - period_risk_free) / overall_volatility if overall_volatility > 0 else 0

print("Portfolio Trades:")
print(portfolio_df[['asset', 'type', 'shares', 'buy_price', 'sell_price', 'cost', 'revenue']])
print(f"\nInvestment Made: ${investment_made:.2f}")
print(f"Remaining Balance: ${balance:.2f}")
print(f"Portfolio Final Balance: ${portfolio_final_balance:.2f}")
print(f"\n--- Metrics over the 150-minute forecast window (not annualized) ---")
print(f"Buy-side return: {buy_metrics['return']*100:.4f}%  |  Sharpe: {buy_metrics['sharpe_ratio']:.4f}")
print(f"Sell-side return: {sell_metrics['return']*100:.4f}%  |  Sharpe: {sell_metrics['sharpe_ratio']:.4f}")
print(f"Overall return: {overall_return*100:.4f}%")
print(f"Overall volatility: {overall_volatility:.6f}")
print(f"Overall Sharpe ratio: {overall_sharpe:.4f}")

portfolio_df.to_csv('portfolio_output.csv', index=False)

Portfolio Trades:
  asset  type  shares   buy_price  sell_price         cost   revenue
0   JNJ   buy      13  270.145599  270.682978  3511.892792  6.985922
1   JPM   buy       1  349.171541  351.077522   349.171541  1.905981
2  AAPL  sell      11  335.467923  335.732839  3693.061226  2.914072
3   XOM  sell      10  161.983495  162.836328  1628.363280  8.528331

Investment Made: $9182.49
Remaining Balance: $817.51
Portfolio Final Balance: $9202.82

--- Metrics over the 150-minute forecast window (not annualized) ---
Buy-side return: 0.2303%  |  Sharpe: 3.4126
Sell-side return: 0.2150%  |  Sharpe: 3.9662
Overall return: 0.2214%
Overall volatility: 0.000843
Overall Sharpe ratio: 2.5574


In [ ]:
files.download('portfolio_output.csv')